# Entrainement des modeles sur GPU (Google Colab)

**Le poste de travail du projet n'a pas de GPU.** Les entrainements
menes en local n'etaient que des tests de chaine (3 epoques, 320 px) :
ils prouvent que le code fonctionne, pas que les modeles sont bons.
Le vrai entrainement se fait ici.

## Reglage obligatoire

Menu *Execution* -> *Modifier le type d'execution* -> Accelerateur
materiel : **GPU (T4)**. Sans cela, ce notebook sera aussi lent que le
poste local.

## Preparer l'archive avant de commencer

Sur le poste local :
```bash
python scripts/preparer_envoi.py
```
Deposez `MODELE_envoi.zip` dans votre Google Drive.

Colab coupe les sessions longues. **Travaillez depuis Drive** : sinon
les poids disparaissent a la coupure. Pour le convoyeur (le plus long),
Kaggle est preferable : voir `entrainement_kaggle.ipynb`.


## 1. Verifier le GPU


In [ ]:
!nvidia-smi


## 2. Installer les dependances


In [ ]:
!pip install -q ultralytics pyyaml
import torch, ultralytics
print('torch', torch.__version__, '| CUDA :', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'Aucun GPU. Execution -> Modifier le type d execution -> GPU (T4)')


## 3. Monter le Drive et deployer le projet


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile

# Chemin de l'archive deposee dans votre Drive
ARCHIVE = '/content/drive/MyDrive/MODELE_envoi.zip'   # <-- a adapter

if os.path.exists(ARCHIVE):
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall('/content/drive/MyDrive/')
    PROJET = '/content/drive/MyDrive/MODELE'
else:
    # Variante : le dossier MODELE est deja dans le Drive
    PROJET = '/content/drive/MyDrive/MODELE'

os.chdir(PROJET)
print('Repertoire de travail :', os.getcwd())
!ls


## 4. Generer les datasets synthetiques

A sauter si vos donnees annotees sont deja dans l'archive.

Le generateur convoyeur produit les **9 classes localisables** de la
taxonomie (le desalignement est traite par la couche vision classique,
ce n'est pas un objet localisable dans l'image).


In [ ]:
!python scripts/generer_dataset_convoyeur.py --nombre 1200
!python scripts/generer_dataset_eclairage.py --nombre 800


Si vous avez des images REELLES de votre convoyeur, meme sans aucun
defaut, la methode hybride est nettement meilleure : tout est reel sauf
la dechirure incrustee. Voir `docs/sans_donnees.md`.


In [ ]:
# !python -m src.prepare.extract_frames --source data/raw/convoyeur.mp4 \
#        --sortie data/frames/convoyeur --intervalle 1
# !python scripts/generer_dechirures_sur_reel.py \
#        --source data/frames/convoyeur --par-image 3


## 5. Controler le dataset

Trois minutes ici evitent d'attendre trois heures un resultat fausse
par une fuite train/val ou une classe vide.


In [ ]:
!python -m src.prepare.check_dataset --modele convoyeur
!python -m src.mlops.registre --empreinte convoyeur


## 6. Entrainer

Un modele a la fois. Le script lit `configs/<modele>.yaml`.

| Modele | Resolution | Epochs | Duree sur T4 |
|--------|-----------|--------|--------------|
| Convoyeur (segmentation, 10 classes) | 1024 | 200 | 3 h a 4 h |
| Eclairage | 960 | 150 | 1 h 45 |
| Vehicules | 640 | 120 | 1 h 15 |

Colab coupe souvent avant 4 h. Deux options pour le convoyeur :
reduire a `--epochs 120`, ou utiliser Kaggle (sessions jusqu'a 12 h).

En cas de `CUDA out of memory`, reduisez `--batch` (8, puis 4).


In [ ]:
!python -m src.train.train --modele convoyeur --epochs 120


In [ ]:
# !python -m src.train.train --modele eclairage


In [ ]:
# !python -m src.train.train --modele vehicules


### Reprendre apres une coupure

Le projet etant sur Drive, `last.pt` est conserve et l'entrainement
repart ou il s'est arrete.


In [ ]:
# !python -m src.train.train --modele convoyeur --reprendre


## 7. Evaluer sur le lot de test

Le lot de test n'a jamais ete vu pendant l'entrainement : c'est le seul
chiffre presentable comme performance reelle dans le rapport.

Regardez le **detail par classe**, pas seulement le mAP global : sur ce
projet, un mAP moyen correct peut cacher deux classes a zero.


In [ ]:
!python -m src.train.evaluer --modele convoyeur --exporter onnx


### Courbes et matrice de confusion pour le rapport


In [ ]:
from IPython.display import Image, display
import glob
for chemin in sorted(glob.glob('runs/convoyeur/train/*.png')):
    print(chemin)
    display(Image(chemin, width=760))


## 8. Promouvoir en production

La promotion verifie que le modele correspond bien aux classes
declarees dans `configs/data_convoyeur.yaml`. Elle **refuse** un modele
entraine sur une autre liste de classes : un modele decale ne plante
pas, il renvoie de mauvais noms de defauts, ce qui est pire.


In [ ]:
!python -m src.mlops.registre --lister
!python -m src.mlops.registre --promouvoir convoyeur --version v1
!ls -lh models/convoyeur/


---
## De retour sur le poste local

Les poids sont dans votre Drive. Copiez le dossier `models/` sur le
poste, puis :

```bash
python -m src.mlops.registre --lister
python -m src.pipeline.run_stream --camera cam_convoyeur_01
```

Le pipeline ne charge que les poids promus, et refuse ceux dont les
classes ne correspondent plus a la configuration.
